# 02 — Bronze Ingestion

**Project:** Incremental Orders Pipeline with Delta Lake  
**Layer:** Bronze  
**Source:** Sample order batch tables  
**Target:** `orders_project.bronze_orders_raw`

This notebook ingests incremental order batches into a Bronze Delta table.

The Bronze layer stores raw incoming order events using append logic. Each batch represents a new arrival of data from the source system.

Bronze keeps the full event history, including repeated `order_id` values with different statuses over time.

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit 


In [0]:
spark.sql("USE SCHEMA orders_project")

In [0]:
dbutils.widgets.dropdown(
    "batch_id",
    "1",
    ["1", "2", "3"],
    "Batch ID"
)

batch_id = int(dbutils.widgets.get("batch_id"))

print(f"Selected batch_id: {batch_id}")

In [0]:
source_table = f"orders_project.source_orders_batch_{batch_id}"

source_df = spark.table(source_table)

display(source_df)

In [0]:
bronze_df = (
    source_df
    .withColumn("bronze_ingestion_ts", current_timestamp())
    .withColumn("source_batch_table", lit(source_table))
)

display(bronze_df)

In [0]:
bronze_table = "orders_project.bronze_orders_raw"

In [0]:
bronze_exists = spark.catalog.tableExists(bronze_table)

if bronze_exists:
    existing_batch_count = (
        spark.table(bronze_table)
        .filter(col("batch_id") == batch_id)
        .count()
    )
else:
    existing_batch_count = 0

print(f"Bronze table exists: {bronze_exists}")
print(f"Existing records for batch_id {batch_id}: {existing_batch_count}")

In [0]:
if existing_batch_count == 0:
    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(bronze_table)

    print(f"Batch {batch_id} ingested successfully into {bronze_table}")
else:
    print(f"Batch {batch_id} was already ingested. Skipping append to avoid duplicates.")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_project.bronze_orders_raw
    ORDER BY batch_id, event_ts
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        batch_id,
        COUNT(*) AS total_records
    FROM orders_project.bronze_orders_raw
    GROUP BY batch_id
    ORDER BY batch_id
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        order_id,
        order_status,
        order_amount,
        batch_id,
        event_ts
    FROM orders_project.bronze_orders_raw
    ORDER BY order_id, event_ts
    """)
)

In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY orders_project.bronze_orders_raw
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)